In [1]:
# RNN & LSTM Next Word Predictor

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Embedding, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import random
import warnings
warnings.filterwarnings('ignore')

In [2]:
sample_text = """The quick brown fox jumps over the lazy dog near the river bank.
The dog barked loudly at the fox as it ran through the green meadow.
Rivers flow silently through the valleys and mountains of the land.
The mountains are covered with snow during the cold winter months.
Winter brings cold winds and frozen rivers to the northern regions.
The northern lights dance across the sky on clear winter nights.
Nights in the forest are filled with sounds of owls and crickets.
Crickets chirp loudly in the warm summer evenings near the pond.
The pond reflects the golden light of the setting sun each evening.
Evening walks along the river are peaceful and calming for the soul.
The soul finds comfort in nature and the beauty of the natural world.
Natural beauty surrounds us in forests fields and flowing streams.
Streams carry water from the mountains down to the valleys below.
Below the mountains the rivers widen and slow as they reach the sea.
The sea is vast and deep full of mysteries and wonderful creatures.
Creatures of the deep ocean live in darkness and silence always.
Always the tides come and go driven by the forces of moon and sun.
The sun rises every morning and sets every evening without fail always.
Singing birds bring joy to the hearts of people in every land.
The land is fertile and rich where rivers flow through the plains.
Plains stretch endlessly under the wide blue sky of the open country.
Country roads wind through hills and valleys connecting small towns.
Towns grow larger over time as people come to seek better lives.
Lives are shaped by the choices we make each and every single day.
Every day brings new challenges and new opportunities for all people.
People work together to build strong communities and happy families.
Families share meals and stories around the fire on cold nights.
Cold nights remind us to stay warm and close to those we love most.
Most people find happiness in simple things like love and friendship.
Friendship is a treasure that grows stronger with time and care.
Care for others is one of the greatest gifts we can offer anyone.
Anyone can make a difference in the world by being kind always.
Always be kind to strangers for you never know their struggles well.
Well chosen words can heal wounds and bring peace to troubled minds.
Minds that seek knowledge grow wiser with each passing year always.
Always keep learning for the world is full of wonder and discovery.
Discovery leads to progress and progress leads to a better world.
The world changes slowly but surely through the efforts of many.
Many small actions together create great changes in the wider world.
Communication allows ideas to travel across oceans and continents fast.
Fast trains and planes carry people to distant corners of the earth.
The earth is our home and we must care for it with great devotion.
Devotion to good causes makes the world a more just and fair place.
Place your trust in those who have shown kindness and integrity always.
Always remember that actions speak louder than any words ever could.
Expression through art connects people across cultures and languages.
Languages differ but the emotions they express are universal to all.
All humans share the same basic needs for love safety and belonging.
Belonging to a community gives people strength and a sense of purpose.
Purpose drives people to achieve great things throughout their lives.
Their lives become meaningful when they serve others with open hearts.
Hearts that are open to love and compassion grow stronger over time.
Time passes quickly when we are engaged in work we truly enjoy doing.
Doing what we love brings happiness and energy to our daily lives.
Lives filled with passion and purpose are rich beyond any measure.
Measure your success not by wealth but by the joy you bring others.
Others will remember you for your kindness long after you are gone.
Develop your mind through reading writing and deep thoughtful reflection.
Reflection on the past helps us make better choices for the future.
The future belongs to those who prepare for it with care and diligence.
Diligence in study and work opens many doors in life for everyone.
Everyone deserves a chance to succeed and live a fulfilling life.
Life is short and precious so spend it doing things that truly matter.
Well being comes from balance between work rest play and reflection.
Reflection in quiet moments reveals what truly matters most in life.
Life teaches us lessons through both joy and sorrow in equal measure.
Measure your days not by productivity alone but by moments of grace.
Grace is found in small gestures of kindness given without expectation.
Step by step we climb the mountains of our dreams and aspirations.
Aspirations fuel our drive to become better versions of ourselves.
People who read widely and think deeply contribute much to society.
Society benefits when each person plays their part with full commitment.
Commitment to a cause greater than oneself brings deep satisfaction.
Satisfaction in simple pleasures is the hallmark of a wise person.
A person of wisdom knows when to speak and when to remain silent.
Silent observation reveals much about the world and human behavior.
Behavior shaped by good values leads to a life of integrity always.
Always strive to be honest even when the truth is difficult to share.
Share your knowledge freely for knowledge grows when it is given away.
Away in the mountains the air is clean and the silence is profound.
Profound thoughts come to us in moments of stillness and quiet rest.
Rest is not a luxury but a necessity for a healthy and balanced life.
Growth happens when we step outside our comfort zones with courage.
Courage is not the absence of fear but acting despite it bravely.
Bravely facing challenges builds character and resilience over time.
Over time small habits compound into large and lasting life changes.
Changes in attitude can transform even the most difficult situation.
Character is who you are when no one else is watching you closely.
Wisdom is the fruit of experience reflection and an open humble mind.
Mind your thoughts for they become your words deeds and your destiny.
Destiny is not fixed but shaped by the choices we make each day.
Each day is a gift and an opportunity to grow learn and love more.
More love in the world begins with one person choosing to be kind.
Kind words and gentle actions ripple out into the world endlessly.
Wonder is the beginning of all great learning and creative discovery.
All journeys begin with a single step taken in faith and courage.
Courage love and wisdom are the three pillars of a good life well lived."""

# Save to Colab's filesystem
with open("/content/sample_text.txt", "w") as f:
    f.write(sample_text)

print("✅ File created successfully at /content/sample_text.txt")

✅ File created successfully at /content/sample_text.txt


In [3]:
def load_text(filepath):
    """Load and preprocess the text file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read().lower()

    # Basic cleanup
    text = ' '.join(text.split())

    print("=" * 60)
    print("STEP 1: TEXT LOADED")
    print("=" * 60)
    print(f"  Total Characters : {len(text)}")
    print(f"  Total Words      : {len(text.split())}")
    print(f"  Sample (first 200 chars):\n  {text[:200]}")
    return text

In [4]:
#CREATE VOCABULARY & VECTORIZE TEXT


def build_vocabulary(text):
    """Create word-level vocabulary and mappings."""
    words = text.split()
    unique_words = sorted(set(words))

    word2idx = {word: idx for idx, word in enumerate(unique_words)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    vocab_size = len(unique_words)

    print("\n" + "=" * 60)
    print("STEP 2: VOCABULARY CREATED")
    print("=" * 60)
    print(f"  Vocabulary Size  : {vocab_size}")
    print(f"  Sample Words     : {unique_words[:10]}")
    print(f"  word2idx sample  : {dict(list(word2idx.items())[:5])}")

    return words, word2idx, idx2word, vocab_size

In [5]:
#CREATE TRAINING SEQUENCES


def create_sequences(words, word2idx, seq_length=5):
    """Create (X, y) sequences for training."""
    sequences = []
    targets   = []

    for i in range(len(words) - seq_length):
        seq    = words[i : i + seq_length]
        target = words[i + seq_length]
        sequences.append([word2idx[w] for w in seq])
        targets.append(word2idx[target])

    X = np.array(sequences)
    y = np.array(targets)

    print("\n" + "=" * 60)
    print("STEP 3: TRAINING SEQUENCES CREATED")
    print("=" * 60)
    print(f"  Sequence Length  : {seq_length}")
    print(f"  Total Sequences  : {len(X)}")
    print(f"  X shape          : {X.shape}")
    print(f"  y shape          : {y.shape}")
    print(f"  Sample sequence  : {X[0]}  →  target: {y[0]}")

    return X, y

In [11]:
#BUILD RNN & LSTM MODELS
from tensorflow.keras.layers import Input

def build_rnn_model(vocab_size, seq_length, embed_dim=64, rnn_units=128):
    model = Sequential([
        Input(shape=(seq_length,)),          # 👈 ADD THIS LINE
        Embedding(input_dim=vocab_size, output_dim=embed_dim),
        SimpleRNN(rnn_units, return_sequences=True),
        Dropout(0.3),
        SimpleRNN(rnn_units),
        Dropout(0.3),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy']
    )
    print("\n" + "=" * 60)
    print("STEP 4a: RNN MODEL BUILT")
    print("=" * 60)
    model.summary()
    return model


def build_lstm_model(vocab_size, seq_length, embed_dim=64, lstm_units=128):
    model = Sequential([
        Input(shape=(seq_length,)),          # 👈 ADD THIS LINE
        Embedding(input_dim=vocab_size, output_dim=embed_dim),
        LSTM(lstm_units, return_sequences=True),
        Dropout(0.3),
        LSTM(lstm_units),
        Dropout(0.3),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy']
    )
    print("\n" + "=" * 60)
    print("STEP 4b: LSTM MODEL BUILT")
    print("=" * 60)
    model.summary()
    return model


In [12]:
#TRAIN THE MODELS

def train_model(model, X, y, model_name="Model", epochs=30, batch_size=64):
    """Train the given model."""
    callbacks = [
        EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, min_lr=1e-5)
    ]

    print("\n" + "=" * 60)
    print(f"STEP 5: TRAINING {model_name.upper()}")
    print("=" * 60)

    history = model.fit(
        X, y,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    final_loss = history.history['loss'][-1]
    final_acc  = history.history['accuracy'][-1]
    print(f"\n   {model_name} Training Complete!")
    print(f"  Final Loss     : {final_loss:.4f}")
    print(f"  Final Accuracy : {final_acc:.4f}")
    return history

In [13]:
#TEXT GENERATION FUNCTION

def generate_text(model, seed_text, word2idx, idx2word,
                  seq_length=5, num_words=50, temperature=1.0):
    """
    Generate text from a seed phrase using the trained model.
    temperature > 1  → more random/creative
    temperature < 1  → more conservative/repetitive
    temperature = 1  → neutral sampling
    """
    # Prepare seed
    words_so_far = seed_text.lower().split()

    # Pad or trim seed to seq_length
    if len(words_so_far) < seq_length:
        words_so_far = ['the'] * (seq_length - len(words_so_far)) + words_so_far

    generated = list(words_so_far[-seq_length:])

    for _ in range(num_words):
        # Build input sequence (handle OOV with 'the')
        input_seq = np.array([[
            word2idx.get(w, word2idx.get('the', 0))
            for w in generated[-seq_length:]
        ]])

        # Get predictions
        preds = model.predict(input_seq, verbose=0)[0]

        # Apply temperature scaling
        preds = np.log(preds + 1e-10) / temperature
        preds = np.exp(preds)
        preds = preds / preds.sum()

        # Sample from distribution
        next_idx = np.random.choice(len(preds), p=preds)
        next_word = idx2word[next_idx]
        generated.append(next_word)

    return ' '.join(generated)

In [9]:
#GENERATE TEXT & VALIDATE

def validate_and_generate(rnn_model, lstm_model, word2idx, idx2word,
                           seq_length=5):
    """Generate text from both models and validate the output."""

    seed_phrases = [
        "the river flows through",
        "knowledge grows when it",
        "the mountains are covered"
    ]

    print("\n" + "=" * 60)
    print("STEP 7: TEXT GENERATION & VALIDATION")
    print("=" * 60)

    for seed in seed_phrases:
        print(f"\n  📌 Seed: '{seed}'")
        print("  " + "-" * 50)

        rnn_text  = generate_text(rnn_model,  seed, word2idx, idx2word,
                                  seq_length=seq_length, num_words=25,
                                  temperature=0.8)
        lstm_text = generate_text(lstm_model, seed, word2idx, idx2word,
                                  seq_length=seq_length, num_words=25,
                                  temperature=0.8)

        print(f"  🔵 RNN  → {rnn_text}")
        print(f"  🟢 LSTM → {lstm_text}")

    # ── Quality Metrics ──────────────────────────────────────
    print("\n" + "=" * 60)
    print("VALIDATION METRICS")
    print("=" * 60)

    seed = "the world changes slowly"
    for name, model in [("RNN", rnn_model), ("LSTM", lstm_model)]:
        gen = generate_text(model, seed, word2idx, idx2word,
                            seq_length=seq_length, num_words=40,
                            temperature=0.7)
        gen_words   = gen.split()
        known_words = sum(1 for w in gen_words if w in word2idx)
        unique_words_gen = len(set(gen_words))

        print(f"\n  Model            : {name}")
        print(f"  Generated text   : {gen}")
        print(f"  Total words      : {len(gen_words)}")
        print(f"  In-vocabulary    : {known_words}/{len(gen_words)}"
              f" ({known_words/len(gen_words)*100:.1f}%)")
        print(f"  Unique words     : {unique_words_gen}")
        print(f"  Diversity ratio  : {unique_words_gen/len(gen_words):.2f}")

In [14]:
# MAIN PIPELINE

if __name__ == "__main__":

    FILE_PATH  = "/content/sample_text.txt"
    SEQ_LENGTH = 5
    EMBED_DIM  = 64
    UNITS      = 128
    EPOCHS     = 40
    BATCH_SIZE = 64

    tf.random.set_seed(42)
    np.random.seed(42)

    # Step 1 – Load
    text = load_text(FILE_PATH)

    # Step 2 – Vocabulary
    words, word2idx, idx2word, vocab_size = build_vocabulary(text)

    # Step 3 – Sequences
    X, y = create_sequences(words, word2idx, seq_length=SEQ_LENGTH)

    # Step 4 – Build Models
    rnn_model  = build_rnn_model(vocab_size,  SEQ_LENGTH, EMBED_DIM, UNITS)
    lstm_model = build_lstm_model(vocab_size, SEQ_LENGTH, EMBED_DIM, UNITS)

    # Step 5 – Train
    rnn_history  = train_model(rnn_model,  X, y, "RNN Model",
                               epochs=EPOCHS, batch_size=BATCH_SIZE)
    lstm_history = train_model(lstm_model, X, y, "LSTM Model",
                               epochs=EPOCHS, batch_size=BATCH_SIZE)

    # Step 6 & 7 – Generate & Validate
    validate_and_generate(rnn_model, lstm_model, word2idx, idx2word,
                          seq_length=SEQ_LENGTH)

    print("\n✅ Pipeline Complete!")

STEP 1: TEXT LOADED
  Total Characters : 6622
  Total Words      : 1138
  Sample (first 200 chars):
  the quick brown fox jumps over the lazy dog near the river bank. the dog barked loudly at the fox as it ran through the green meadow. rivers flow silently through the valleys and mountains of the land

STEP 2: VOCABULARY CREATED
  Vocabulary Size  : 534
  Sample Words     : ['a', 'about', 'absence', 'achieve', 'across', 'acting', 'actions', 'after', 'air', 'all']
  word2idx sample  : {'a': 0, 'about': 1, 'absence': 2, 'achieve': 3, 'across': 4}

STEP 3: TRAINING SEQUENCES CREATED
  Sequence Length  : 5
  Total Sequences  : 1133
  X shape          : (1133, 5)
  y shape          : (1133,)
  Sample sequence  : [464 377  61 197 253]  →  target: 345

STEP 4a: RNN MODEL BUILT


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 5, 64)          │        34,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 5, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 534)            │        68,886 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,662 (627.59 KB)

 Trainable params: 160,662 (627.59 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 5, 64)          │        34,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 5, 128)         │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 534)            │        68,886 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333,462 (1.27 MB)

 Trainable params: 333,462 (1.27 MB)

 Non-trainable params: 0 (0.00 B)


STEP 5: TRAINING RNN MODEL
Epoch 1/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.0142 - loss: 6.2618 - learning_rate: 0.0010
Epoch 2/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0550 - loss: 5.7984 - learning_rate: 0.0010
Epoch 3/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0643 - loss: 5.5747 - learning_rate: 0.0010
Epoch 4/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0714 - loss: 5.4521 - learning_rate: 0.0010
Epoch 5/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0940 - loss: 5.2813 - learning_rate: 0.0010
Epoch 6/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1204 - loss: 4.9589 - learning_rate: 0.0010
Epoch 7/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1856 - loss: 4.6366 - learning_rate: 0.0010
Epoch 8/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2208 - loss: 4.3170 - learning_rate: 0.0010
Epoch 9/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.2785 - loss: 3.9812 - learning_rate

RNN (Recurrent Neural Network) processes sequential data by maintaining a hidden state across time steps, but suffers from the vanishing gradient problem for long sequences. LSTM (Long Short-Term Memory) solves this using three gates — forget, input, and output — along with a cell state, allowing it to retain long-term dependencies. Both models use an Embedding layer to represent words as dense vectors and temperature sampling to control randomness during text generation.


The dataset contained 338 unique words with 1133 training sequences of length 5.
Both RNN and LSTM successfully generated text in the style of the training data with 100% in-vocabulary output.
RNN converged faster but showed repetitive patterns in generated text, a direct effect of vanishing gradients.
LSTM produced more contextually consistent and diverse sentences due to its gated memory mechanism.
EarlyStopping and ReduceLROnPlateau callbacks helped both models avoid overfitting.


LSTM outperforms SimpleRNN for next-word prediction as it retains longer context. Even on a small corpus, the gated architecture of LSTM resulted in better text quality, higher diversity ratio, and smoother sentence flow compared to RNN.